# Phase 7 — Prior Case Retrieval with the pre-trained **Encoder–Decoder** (bidirectional encoder)

Given a query case and its split's candidate pool, return the 7 most relevant prior cases. The metric is **Micro-F1@7**. The Phase-1 **encoder** becomes a **bi-encoder** (the decoder is dropped): every case is embedded independently, and relevance is a similarity between the embeddings. A pool is encoded once, and ranking is a matrix product.

```
case → 511-token chunks + </s> (8 spread over the judgment at test time; 4 random ones per case during training)
     → bidirectional encoder (adapted) → final states of each chunk
     → attention pooling over tokens (zero-init = mean) → projection (identity init) → unit chunk vectors
     → attention aggregation over chunks (zero-init = mean) → one unit vector per case
score(query, candidate) = cosine            (or "maxsim": chunk-level late interaction)
```

Because of the identity and zero initialisations, the **untrained retriever is exactly mean-pooled Phase-1 encoder states**. `D0_zeroshot` measures how Phase-1 representations transfer, and every method starts from that point.

**Training:** InfoNCE over the batch. Each query has 1 relevant case and 3 hard negatives (the top BM25 non-relevant cases from its pool), and every other case in the batch is an in-batch negative. Cases relevant to the query are never used as negatives. Long cases make a 40-case batch about 160 chunk forwards, so the notebook uses **GradCache** (Gao et al., 2021): the exact full-batch gradient with the memory of one sub-batch. A self-test verifies this against ordinary backprop.

**Techniques (the Phase-7 list):** `full` · `partial` · `lorma` (LoRMA⁺, compulsory) · `ia3` · `prefix` · `prompt` · `qlora` · `kd` (half-depth encoder student initialised from the fine-tuned teacher). The KD student learns from the teacher's score distribution over each batch (listwise KL), from the teacher's case vectors (cosine alignment), and from the teacher's hard negatives. `lora` is available as an extra.

**BM25** over our own BPE ids (unigrams + hashed bigrams) is computed for three things: a lexical reference, hard negatives, and an optional *hybrid* (z-normalised dense + BM25, weight tuned on dev). Hybrid results are reported and written as separate `_hybrid` submissions. Every system still ranks with the fine-tuned Phase-1 model.

**Run order:** top to bottom. The checks cell verifies the encoder runner against the Phase-1 encoder, PEFT identity at initialisation, and GradCache against direct gradients. The run cell starts with BM25, the zero-shot model and a smoke test.

In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
import os, json, math, time, random, copy, pickle, glob, shutil, gc
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_ckpt

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")
AMP_DTYPE = (None if device.type != "cuda" else
             torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)   # fp16 on T4
print("mixed precision:", AMP_DTYPE)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
print("HF token available:", bool(HF_TOKEN))

# ---- paths (same layout as the Phase-5 encoder-decoder notebook; edit if yours differ) --------------
DRIVE_ROOT = "/content/drive/MyDrive/Phase1_Pretraining"
WORK_DIR   = os.path.join(DRIVE_ROOT, "phase1_encoder_decoder")
CKPT_DIR   = os.path.join(WORK_DIR, "checkpoints")
ckpt_path  = os.path.join(CKPT_DIR, "encoder_decoder_best.pt")
if not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(CKPT_DIR, "encoder_decoder_latest.pt")
tok_path   = os.path.join(WORK_DIR, "final_bpe_en.json")
p7_dir     = os.path.join(WORK_DIR, "phase7"); os.makedirs(p7_dir, exist_ok=True)
local_ck   = "/content/p7_ckpt"; os.makedirs(local_ck, exist_ok=True)

PHASE1_NB = None
for pat in ("/content/drive/MyDrive/Colab Notebooks/phase1_encoder_decoder*.ipynb",
            os.path.join(WORK_DIR, "*.ipynb"), "/content/phase1_encoder_decoder*.ipynb"):
    hits = sorted(glob.glob(pat))
    if hits:
        PHASE1_NB = hits[0]; break
for what, p in (("checkpoint", ckpt_path), ("tokenizer", tok_path)):
    if not os.path.exists(p):
        raise FileNotFoundError(f"Phase-1 {what} not found: {p}  -> edit the paths above")
print("checkpoint:", ckpt_path, "| phase-1 notebook:", PHASE1_NB)


def load_definitions_from_notebook(nb_path):
    """exec the Phase-1 cells that define BPETokenizer and normalize_text (identical tokenisation)."""
    if nb_path is None:
        raise FileNotFoundError("Phase-1 notebook not found: set PHASE1_NB")
    nb = json.load(open(nb_path, encoding="utf-8"))
    picked = []
    for marker in ("class BPETokenizer", "def normalize_text"):
        for i, c in enumerate(nb["cells"]):
            if c["cell_type"] == "code" and marker in "".join(c["source"]) and i not in picked:
                picked.append(i); break
        else:
            raise RuntimeError(f"no cell containing {marker!r} in {nb_path}")
    for i in sorted(picked):
        src = "".join(nb["cells"][i]["source"])
        src = "\n".join(l for l in src.split("\n") if not l.lstrip().startswith(("!", "%")))
        exec(compile(src, f"<phase1 cell {i}>", "exec"), globals())
    print("loaded definitions from Phase-1 cells", sorted(picked))


if not all(n in globals() for n in ("BPETokenizer", "normalize_text")):
    load_definitions_from_notebook(PHASE1_NB)

tokenizer = BPETokenizer.from_json(tok_path)
PAD_ID, BOS_ID, EOS_ID = tokenizer.stoi["<pad>"], tokenizer.stoi["<s>"], tokenizer.stoi["</s>"]
print(f"tokenizer vocab: {len(tokenizer.itos):,} | PAD {PAD_ID}  BOS {BOS_ID}  EOS {EOS_ID}")

Mounted at /content/drive
device: cuda | Tesla T4
mixed precision: torch.float16
HF token available: True
checkpoint: /content/drive/MyDrive/Phase1_Pretraining/phase1_encoder_decoder/checkpoints/encoder_decoder_best.pt | phase-1 notebook: /content/drive/MyDrive/Colab Notebooks/phase1_encoder_decoder_bpe_pretraining_txl_data_pipeline-2.ipynb
loaded definitions from Phase-1 cells [7, 13]
tokenizer vocab: 32,768 | PAD 0  BOS 2  EOS 3


In [ ]:
# ============================================================
# 1. PRE-TRAINED ENCODER-DECODER  (architecture copied from Phase 1 so the state-dict loads strictly)
#
# Only two changes, both numerically equivalent to the original:
#   * Encoder.forward folds [relative-position bias + key-padding mask] into ONE additive bias, built once
#     per forward and shared by all layers, and attention uses F.scaled_dot_product_attention (fused,
#     no softmax matrix kept). FAST_ATTN["on"]=False restores the original manual attention.
#   * Encoder.forward(n_frozen=k): the first k layers run under no_grad (frozen part of a partly fine-tuned encoder).
# The decoder is untouched (we only ever run it for ONE query token, so it is cheap).
# ============================================================
FAST_ATTN = {"on": True}


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        var = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(var + self.eps)
        return self.weight * x


class RelativePositionBias(nn.Module):
    """T5-style relative position bias, bucketed on a log scale for far-apart positions."""

    def __init__(self, num_heads, num_buckets=32, max_distance=128, bidirectional=True):
        super().__init__()
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(num_buckets, num_heads)

    @staticmethod
    def _bucket(relative_position, bidirectional, num_buckets, max_distance):
        ret = 0
        n = -relative_position
        if bidirectional:
            num_buckets //= 2
            ret += (n < 0).to(torch.long) * num_buckets
            n = torch.abs(n)
        else:
            n = torch.clamp(n, min=0)
        max_exact = num_buckets // 2
        is_small = n < max_exact
        val_if_large = max_exact + (
            torch.log(n.float() / max_exact + 1e-6)
            / math.log(max_distance / max_exact)
            * (num_buckets - max_exact)
        ).to(torch.long)
        val_if_large = torch.clamp(val_if_large, max=num_buckets - 1)
        return ret + torch.where(is_small, n, val_if_large)

    def forward(self, q_len, k_len, device):
        q_pos = torch.arange(q_len, device=device)[:, None]
        k_pos = torch.arange(k_len, device=device)[None, :]
        rel_pos = k_pos - q_pos
        bucket = self._bucket(rel_pos, self.bidirectional, self.num_buckets, self.max_distance)
        values = self.embedding(bucket)                    # q_len, k_len, num_heads
        return values.permute(2, 0, 1).unsqueeze(0)        # 1, num_heads, q_len, k_len


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.q = nn.Linear(d_model, d_model, bias=False)
        self.k = nn.Linear(d_model, d_model, bias=False)
        self.v = nn.Linear(d_model, d_model, bias=False)
        self.o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x_q, x_kv, mask=None, position_bias=None, attn_bias=None):
        B, Lq, D = x_q.shape
        Lk = x_kv.shape[1]
        q = self.q(x_q).view(B, Lq, self.num_heads, self.d_head).transpose(1, 2)
        k = self.k(x_kv).view(B, Lk, self.num_heads, self.d_head).transpose(1, 2)
        v = self.v(x_kv).view(B, Lk, self.num_heads, self.d_head).transpose(1, 2)

        if attn_bias is not None:            # fused path: softmax(q k^T / sqrt(d_head) + attn_bias) v
            out = F.scaled_dot_product_attention(
                q, k, v, attn_mask=attn_bias.to(q.dtype),
                dropout_p=self.dropout.p if self.training else 0.0)
        else:                                # original path
            scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
            if position_bias is not None:
                scores = scores + position_bias
            if mask is not None:
                scores = scores.masked_fill(mask == 0, float("-inf"))
            attn = self.dropout(torch.softmax(scores, dim=-1))
            out = torch.matmul(attn, v)
        return self.o(out.transpose(1, 2).contiguous().view(B, Lq, D))


class GatedFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.wi_0 = nn.Linear(d_model, d_ff, bias=False)
        self.wi_1 = nn.Linear(d_model, d_ff, bias=False)
        self.wo = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()

    def forward(self, x):
        h = self.act(self.wi_0(x)) * self.wi_1(x)
        return self.wo(self.dropout(h))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln2 = RMSNorm(d_model)
        self.ff = GatedFeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask, position_bias, attn_bias=None):
        h = self.ln1(x)
        x = x + self.dropout(self.self_attn(h, h, mask, position_bias, attn_bias))
        h = self.ln2(x)
        x = x + self.dropout(self.ff(h))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln2 = RMSNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln3 = RMSNorm(d_model)
        self.ff = GatedFeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, self_mask, cross_mask, self_pos_bias):
        h = self.ln1(x)
        x = x + self.dropout(self.self_attn(h, h, self_mask, self_pos_bias))
        h = self.ln2(x)
        x = x + self.dropout(self.cross_attn(h, enc_out, cross_mask, None))
        h = self.ln3(x)
        x = x + self.dropout(self.ff(h))
        return x


class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.pos_bias = RelativePositionBias(num_heads, bidirectional=True)
        self.final_ln = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask, n_frozen=0):
        L = x.shape[1]
        pos_bias = self.pos_bias(L, L, x.device)                      # [1,H,L,L]
        mask = attention_mask[:, None, None, :]                       # [B,1,1,L]
        attn_bias = None
        if FAST_ATTN["on"]:
            with torch.no_grad():                                     # dtype the attention will run in (fp32 or autocast)
                dt = self.layers[0].self_attn.q(x[:, :1]).dtype
            keep = attention_mask.bool()[:, None, None, :]
            attn_bias = pos_bias.to(dt).expand(x.size(0), -1, -1, -1).masked_fill(~keep, float("-inf"))   # [B,H,L,L], built once, shared by all layers
        x = self.dropout(x)
        for i, layer in enumerate(self.layers):
            if i < n_frozen:                                          # frozen bottom layers: no graph, no memory
                with torch.no_grad():
                    x = layer(x, mask, pos_bias, attn_bias)
            else:
                x = layer(x, mask, pos_bias, attn_bias)
        return self.final_ln(x)


class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.pos_bias = RelativePositionBias(num_heads, bidirectional=False)
        self.final_ln = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, self_attention_mask, cross_attention_mask):
        L = x.shape[1]
        causal = torch.tril(torch.ones(L, L, device=x.device, dtype=torch.bool))
        pad = self_attention_mask[:, None, None, :].bool()
        self_mask = (causal[None, None] & pad).float()
        cross_mask = cross_attention_mask[:, None, None, :]
        pos_bias = self.pos_bias(L, L, x.device)
        x = self.dropout(x)
        for layer in self.layers:
            x = layer(x, enc_out, self_mask, cross_mask, pos_bias)
        return self.final_ln(x)


class LegalSeq2SeqLM(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=16, d_ff=2048,
                 num_encoder_layers=9, num_decoder_layers=9, dropout=0.1, pad_id=0):
        super().__init__()
        self.shared_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_encoder_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_decoder_layers, dropout)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.shared_embed.weight  # tied embeddings

    def forward(self, input_ids, attention_mask, decoder_input_ids, decoder_attention_mask):
        enc_out = self.encoder(self.shared_embed(input_ids), attention_mask)
        dec_out = self.decoder(
            self.shared_embed(decoder_input_ids), enc_out, decoder_attention_mask, attention_mask
        )
        return self.lm_head(dec_out)


def load_pretrained_ed():
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    c = ck.get("model_config", {})
    m = LegalSeq2SeqLM(
        vocab_size=c.get("vocab_size", len(tokenizer.itos)), d_model=c.get("d_model", 512),
        num_heads=c.get("num_heads", 16), d_ff=c.get("d_ff", 2048),
        num_encoder_layers=c.get("num_encoder_layers", 9), num_decoder_layers=c.get("num_decoder_layers", 9),
        dropout=c.get("dropout", 0.1), pad_id=c.get("pad_id", PAD_ID))
    if m.shared_embed.num_embeddings != len(tokenizer.itos):
        raise ValueError(f"model vocab {m.shared_embed.num_embeddings} != tokenizer vocab {len(tokenizer.itos)}: "
                         f"wrong tokenizer file for this checkpoint")
    m.load_state_dict(ck["model_state_dict"])            # strict: fails loudly on any mismatch
    print(f"loaded Phase-1 checkpoint | step {ck.get('global_step')} | best val token-loss {ck.get('best_val_loss')}")
    return m.eval()


ED = load_pretrained_ed().to(device)
D_ED = ED.shared_embed.embedding_dim
print(f"Encoder-decoder loaded: {sum(p.numel() for p in ED.parameters())/1e6:.1f}M params | d_model {D_ED} | "
      f"{len(ED.encoder.layers)} enc + {len(ED.decoder.layers)} dec layers")

loaded Phase-1 checkpoint | step 11200 | best val token-loss 4.871949663513102
Encoder-decoder loaded: 101.7M params | d_model 512 | 9 enc + 9 dec layers


In [ ]:
# ============================================================
# Small utilities: seeds, autocast, chunking, RNG capture (for GradCache), checkpoint I/O
# ============================================================
import collections


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def amp_ctx(on=True):
    return torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=bool(on and AMP_DTYPE is not None))


def chunk_ids(ids, body_len=511, stride=511, min_tail=16):
    if len(ids) <= body_len:
        return [ids]
    chunks, s = [], 0
    while True:
        chunks.append(ids[s:s + body_len])
        if s + body_len >= len(ids):
            break
        s += stride
    if len(chunks) > 1 and len(chunks[-1]) < min_tail:
        chunks.pop()
    return chunks


def select_chunks(chunks, max_chunks, strategy="uniform"):
    n = len(chunks)
    if n <= max_chunks:
        return chunks
    if strategy == "head":
        return chunks[:max_chunks]
    if strategy == "head_tail":
        t = max_chunks // 2
        return chunks[:max_chunks - t] + (chunks[-t:] if t else [])
    if strategy == "uniform":
        idx = np.unique(np.linspace(0, n - 1, max_chunks).round().astype(int))
        return [chunks[i] for i in idx]
    raise ValueError(strategy)


def get_rng():
    return (torch.get_rng_state(), torch.cuda.get_rng_state() if torch.cuda.is_available() else None)


def set_rng(st):
    torch.set_rng_state(st[0])
    if st[1] is not None:
        torch.cuda.set_rng_state(st[1])


def save_trainable(m, path, extra=None):
    torch.save({"trainable": {n: p.detach().cpu().clone() for n, p in m.named_parameters() if p.requires_grad},
                **(extra or {})}, path)


def load_trainable(m, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    missing = [n for n, p in m.named_parameters() if p.requires_grad and n not in ck["trainable"]]
    if missing:
        raise RuntimeError(f"{path} lacks {len(missing)} trainable tensors, e.g. {missing[:3]}")
    m.load_state_dict(ck["trainable"], strict=False)
    return ck

In [ ]:
# ============================================================
# DATA: Phase-7 Prior Case Retrieval (query cases + one candidate pool per split)
#   Train/train.json + train_candidates.json | Dev/dev.json + dev_candidates.json | Test/test.json + test_candidates.json
#   queries: id, text (list of sentences), relevant_candidates (not in test) | candidates: id, text
# ============================================================
from huggingface_hub import hf_hub_download, HfApi

REPO_ID = "Exploration-Lab/CS-781-Capstone"
_REPO_FILES = HfApi().list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN)


def find_repo_file(basename):
    hits = [f for f in _REPO_FILES if f.endswith("/" + basename) and ("pcr" in f.lower() or "phase_7" in f.lower())]
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {basename} for Phase 7, got {hits}. Phase-7-like files: "
                           f"{[f for f in _REPO_FILES if 'pcr' in f.lower()]}")
    return hits[0]


_TEXT_KEYS = ("text", "document", "judgment", "judgement", "sentences", "doc")
_REL_KEYS = ("relevant_candidates", "relevant_cases", "relevant", "positives")
_CAND_KEYS = ("candidates", "candidate_ids", "candidate_pool")    # per-query candidate list, if the data has one


def _as_text(v):
    if v is None:
        return ""
    if isinstance(v, list):
        return " ".join(str(s).strip() for s in v if str(s).strip())
    return str(v).strip()


def _norm_id(x):
    return str(x).strip()


def _records(raw):
    if isinstance(raw, dict):                              # {id: {...}} or {id: text}
        return [({**v, "id": k} if isinstance(v, dict) else {"id": k, "text": v}) for k, v in raw.items()]
    return raw


def read_json(filename):
    path = hf_hub_download(REPO_ID, find_repo_file(filename), repo_type="dataset", token=HF_TOKEN)
    return _records(json.load(open(path, encoding="utf-8")))


def read_queries(filename):
    raw = read_json(filename)
    keys = list(raw[0].keys())
    tk = next(k for k in _TEXT_KEYS if k in keys)
    rk = next((k for k in _REL_KEYS if k in keys), None)
    ck = next((k for k in _CAND_KEYS if k in keys and isinstance(raw[0][k], list)), None)
    print(f"{filename}: {len(raw)} queries | fields {keys} -> text '{tk}', relevance '{rk}', per-query candidates '{ck}'")
    if rk is None and not filename.startswith("test"):
        raise KeyError(f"no relevance field in {filename} among {_REL_KEYS}: add its name from the fields above")
    return [dict(id=_norm_id(r["id"]), text=_as_text(r[tk]),
                 rel=None if rk is None else [_norm_id(c) for c in (r.get(rk) or [])],
                 cands=None if ck is None else [_norm_id(c) for c in r[ck]]) for r in raw]


def read_pool(filename):
    raw = read_json(filename)
    keys = list(raw[0].keys())
    tk = next(k for k in _TEXT_KEYS if k in keys)
    print(f"{filename}: {len(raw)} candidates | fields {keys} -> text '{tk}'")
    pool = {}
    for r in raw:
        pool[_norm_id(r["id"])] = _as_text(r[tk])
    return pool


SPL = {}
for split, d in (("train", "train"), ("dev", "dev"), ("test", "test")):
    SPL[split] = dict(queries=read_queries(f"{d}.json"), pool=read_pool(f"{d}_candidates.json"))

# ---- sanity + statistics ---------------------------------------------------------------------------------------
for split in ("train", "dev", "test"):
    qs, pool = SPL[split]["queries"], SPL[split]["pool"]
    self_in_pool = sum(q["id"] in pool for q in qs)
    msg = f"[{split}] {len(qs)} queries | pool {len(pool)} | queries that are also in their pool: {self_in_pool}"
    if qs[0]["rel"] is not None:
        n_rel = np.array([len(q["rel"]) for q in qs])
        missing = sum(c not in pool for q in qs for c in q["rel"])
        msg += (f" | relevant/query: mean {n_rel.mean():.2f} median {np.median(n_rel):.0f} max {n_rel.max()} "
                f"| zero-relevant queries {(n_rel == 0).sum()} | relevant ids missing from pool {missing}")
        best_f1 = 2 * np.minimum(n_rel, 7).sum() / (7 * len(qs) + n_rel.sum())
        msg += f" | oracle Micro-F1@7 {best_f1:.3f}"
    print(msg)
print("pool overlap train/dev:", len(set(SPL["train"]["pool"]) & set(SPL["dev"]["pool"])),
      "| train/test:", len(set(SPL["train"]["pool"]) & set(SPL["test"]["pool"])))
_rel_cnt = collections.Counter(c for q in SPL["train"]["queries"] for c in q["rel"])
print("train: most-cited candidates are relevant to", [n for _, n in _rel_cnt.most_common(5)], "queries")


# ---- tokenise every unique document once (cached on Drive) ------------------------------------------------------
def _tok_cached(texts, tag, bs=500, workers=None):
    workers = workers or min(4, os.cpu_count() or 1)
    path = os.path.join(p7_dir, f"tok_{tag}.pkl")
    if os.path.exists(path):
        return pickle.load(open(path, "rb"))
    out = []
    for i in range(0, len(texts), bs):
        part = [normalize_text(t) for t in texts[i:i + bs]]
        out += [np.asarray(x, dtype=np.int32) for x in
                tokenizer.encode_batch(part, add_special_tokens=False, num_workers=workers, chunk_size=8)]
        print(f"  tokenised {min(i + bs, len(texts)):,}/{len(texts):,} ({tag})")
    pickle.dump(out, open(path, "wb"))
    return out


TOK = {}                    # (split, "q"|"c", id) -> int32 token array
for split in ("train", "dev", "test"):
    qs, pool = SPL[split]["queries"], SPL[split]["pool"]
    qt = _tok_cached([q["text"] for q in qs], f"{split}_queries")
    pids = list(pool)
    ct = _tok_cached([pool[i] for i in pids], f"{split}_pool")
    for q, t in zip(qs, qt):
        TOK[(split, "q", q["id"])] = t
    for i, t in zip(pids, ct):
        TOK[(split, "c", i)] = t
    SPL[split]["pool_ids"] = pids
    SPL[split]["qids"] = [q["id"] for q in qs]
    SPL[split]["rel"] = {q["id"]: set(q["rel"]) & set(pool) for q in qs} if qs[0]["rel"] is not None else None
    # allowed candidates per query: the per-query list if the data has one, else the whole split pool
    SPL[split]["allowed"] = ({q["id"]: [c for c in q["cands"] if c in pool] for q in qs} if qs[0]["cands"] is not None
                             else None)

_lens = np.array([len(TOK[("train", "c", i)]) for i in SPL["train"]["pool_ids"]])
_qlens = np.array([len(TOK[("train", "q", i)]) for i in SPL["train"]["qids"]])
pct = lambda a: "  ".join(f"p{p} {int(np.percentile(a, p))}" for p in (10, 50, 90, 99))
print("train candidate BPE tokens:", pct(_lens), f"| 512-chunks median {np.median(np.ceil(_lens / 511)):.0f}")
print("train query     BPE tokens:", pct(_qlens))

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

train.json: 827 queries | fields ['text', 'relevant_candidates', 'id'] -> text 'text', relevance 'relevant_candidates', per-query candidates 'None'


Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

train_candidates.json: 4320 candidates | fields ['text', 'id'] -> text 'text'


... (6 outputs left out)

test.json: 237 queries | fields ['text', 'id'] -> text 'text', relevance 'None', per-query candidates 'None'


Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test_candidates.json: 1727 candidates | fields ['text', 'id'] -> text 'text'
[train] 827 queries | pool 4320 | queries that are also in their pool: 827 | relevant/query: mean 6.42 median 4 max 52 | zero-relevant queries 10 | relevant ids missing from pool 0 | oracle Micro-F1@7 0.639
[dev] 118 queries | pool 1023 | queries that are also in their pool: 118 | relevant/query: mean 8.25 median 5 max 92 | zero-relevant queries 0 | relevant ids missing from pool 0 | oracle Micro-F1@7 0.618
[test] 237 queries | pool 1727 | queries that are also in their pool: 237
pool overlap train/dev: 436 | train/test: 740
train: most-cited candidates are relevant to [14, 10, 10, 9, 8] queries
  tokenised 500/827 (train_queries)
  tokenised 827/827 (train_queries)
  tokenised 500/4,320 (train_pool)
  tokenised 1,000/4,320 (train_pool)
  tokenised 1,500/4,320 (train_pool)
  tokenised 2,000/4,320 (train_pool)
  tokenised 2,500/4,320 (train_pool)
  tokenised 3,000/4,320 (train_pool)
  tokenised 3,500/4,320 (tra

In [ ]:
# ============================================================
# Ranking utilities, Micro-F1@7, submission writer, and a BM25 reference ranker over BPE ids
#
# BM25 is NOT a pretrained model: it is a term-weighting formula over our own tokenizer's ids. It is used for
#   (a) a lexical reference point for the report, (b) hard-negative mining, (c) optional dense+lexical fusion.
# Every submitted system still ranks with the fine-tuned Phase-1 model; fusion is reported separately.
# ============================================================
import scipy.sparse as sp
import zipfile

TOPK = 7


def valid_mask(split):
    """[nq, nc] bool: candidate allowed for that query (same split pool, per-query list if any, never itself)"""
    qids, pids = SPL[split]["qids"], SPL[split]["pool_ids"]
    col = {p: j for j, p in enumerate(pids)}
    allowed = SPL[split].get("allowed")
    if allowed is None:
        m = np.ones((len(qids), len(pids)), dtype=bool)
    else:
        m = np.zeros((len(qids), len(pids)), dtype=bool)
        for i, q in enumerate(qids):
            m[i, [col[c] for c in allowed[q]]] = True
    for i, q in enumerate(qids):
        if q in col:
            m[i, col[q]] = False
    return m


def rank_ids(S, split, k=TOPK):
    """S [nq, nc] scores (numpy) -> {qid: [top-k candidate ids]} (invalid candidates never returned)"""
    S = np.where(valid_mask(split), S, -np.inf)
    k_eff = min(k, S.shape[1])
    top = np.argpartition(-S, k_eff - 1, axis=1)[:, :k_eff]
    order = np.take_along_axis(top, np.argsort(-np.take_along_axis(S, top, 1), axis=1, kind="stable"), 1)
    pids, qids = SPL[split]["pool_ids"], SPL[split]["qids"]
    return {q: [pids[j] for j in order[i] if np.isfinite(S[i, j])] for i, q in enumerate(qids)}


def micro_at_k(pred, rel, k=TOPK):
    hits = n_pred = n_gold = 0
    for q, gold in rel.items():
        p = pred.get(q, [])[:k]
        hits += len(set(p) & gold); n_pred += k; n_gold += len(gold)
    P = hits / max(n_pred, 1); R = hits / max(n_gold, 1)
    return dict(f1=0.0 if P + R == 0 else 2 * P * R / (P + R), p=P, r=R)


def full_metrics(S, split):
    """Micro-P/R/F1@7 plus Recall@50/100 and MRR (for analysis)"""
    rel = SPL[split]["rel"]
    m = micro_at_k(rank_ids(S, split, TOPK), rel)
    Sm = np.where(valid_mask(split), S, -np.inf)
    order = np.argsort(-Sm, axis=1, kind="stable")[:, :100]
    pids = SPL[split]["pool_ids"]
    r50 = r100 = mrr = 0.0; n = 0
    for i, q in enumerate(SPL[split]["qids"]):
        g = rel[q]
        if not g:
            continue
        ids = [pids[j] for j in order[i]]
        r50 += len(set(ids[:50]) & g) / len(g); r100 += len(set(ids) & g) / len(g)
        mrr += next((1 / (r + 1) for r, c in enumerate(ids) if c in g), 0.0); n += 1
    m.update(r50=r50 / max(n, 1), r100=r100 / max(n, 1), mrr=mrr / max(n, 1))
    return m


def fmt_m(m):
    return (f"F1@7 {m['f1']:.4f} (P {m['p']:.4f} R {m['r']:.4f})"
            + (f" | R@50 {m['r50']:.3f} R@100 {m['r100']:.3f} MRR {m['mrr']:.3f}" if "r50" in m else ""))


def write_submission(S, name, split="test"):
    pred = rank_ids(S, split, TOPK)
    pool = set(SPL[split]["pool_ids"])
    for q in SPL[split]["qids"]:
        p = pred[q]
        assert len(p) == TOPK and len(set(p)) == TOPK and all(c in pool for c in p), f"bad prediction for {q}: {p}"
    d = os.path.join(p7_dir, f"submission_{name}"); os.makedirs(d, exist_ok=True)
    jp = os.path.join(d, "prediction.json")
    json.dump(pred, open(jp, "w"), indent=1)
    zp = os.path.join(d, "submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")
    print(f"wrote {len(pred)} queries x {TOPK} ids -> {zp}")
    return zp


# ---------------- BM25 over BPE ids (unigrams + hashed bigrams) ----------------
V_TOK = len(tokenizer.itos)
_HASH = 1 << 22


def _terms(ids, ngram):
    ids = np.asarray(ids, dtype=np.int64)
    t = [ids]
    if ngram >= 2 and len(ids) > 1:
        t.append(V_TOK + (ids[:-1] * 1000003 + ids[1:]) % _HASH)
    return np.unique(np.concatenate(t), return_counts=True)


def _count_matrix(docs, ngram):
    rows, cols, vals = [], [], []
    for i, d in enumerate(docs):
        u, c = _terms(d, ngram)
        rows.append(np.full(len(u), i)); cols.append(u); vals.append(c)
    return sp.csr_matrix((np.concatenate(vals).astype(np.float32), (np.concatenate(rows), np.concatenate(cols))),
                         shape=(len(docs), V_TOK + _HASH))


def bm25_scores(split, k1=1.2, b=0.75, ngram=2, max_df=0.5, query_tf=False):
    """-> [nq, nc] float32 BM25 scores of the split's queries against the split's pool"""
    key = (split, k1, b, ngram, max_df, query_tf)
    if key in _BM25:
        return _BM25[key]
    C = _count_matrix([TOK[(split, "c", i)] for i in SPL[split]["pool_ids"]], ngram)
    Q = _count_matrix([TOK[(split, "q", i)] for i in SPL[split]["qids"]], ngram)
    N = C.shape[0]
    df = np.asarray((C > 0).sum(0)).ravel()
    idf = np.log1p((N - df + 0.5) / (df + 0.5)).astype(np.float32)
    idf[df > max_df * N] = 0.0                                   # stop-word-like pieces
    dl = np.asarray(C.sum(1)).ravel(); avg = dl.mean()
    C = C.tocoo()
    denom = C.data + k1 * (1 - b + b * dl[C.row] / avg)
    W = sp.csr_matrix((idf[C.col] * C.data * (k1 + 1) / denom, (C.row, C.col)), shape=C.shape)
    Qw = Q.copy()
    if not query_tf:
        Qw.data[:] = 1.0
    else:
        Qw.data = Qw.data / (Qw.data + 1.2) * 2.2
    S = (Qw @ W.T).toarray().astype(np.float32)
    _BM25[key] = S
    return S


_BM25 = {}


def zrow(S):
    """per-query z-normalisation over valid candidates (for score fusion)"""
    S = S.astype(np.float64)
    mu, sd = S.mean(1, keepdims=True), S.std(1, keepdims=True) + 1e-9
    return (S - mu) / sd


def tune_fusion(S_dense, split="dev", lex=None, grid=np.linspace(0, 1, 11)):
    """w * z(dense) + (1 - w) * z(BM25): best w on the labelled split"""
    lex = bm25_scores(split) if lex is None else lex
    zd, zl = zrow(S_dense), zrow(lex)
    res = [(w, micro_at_k(rank_ids(w * zd + (1 - w) * zl, split), SPL[split]["rel"])["f1"]) for w in grid]
    return max(res, key=lambda x: x[1]), res

In [ ]:
# ============================================================
# Parameter-efficient adaptation modules (own implementations, no peft library)
#   LoRA     (Hu et al., 2021)            W x + (a/r) B A x                      B = 0 at init
#   LoRMA+   (Bihany, Patel, Modi, 2025)   (I + (a/r) B A) W0 x                   multiplicative, additive rank
#                                          inflation; B = 0 at init -> identity; computed as y0 + s*B(A y0),
#                                          y0 = W0 x, so no d x d matrix is ever formed (the paper's re-ordering)
#   QLoRA    (Dettmers et al., 2023)       frozen base stored as 4-bit NF4 (blockwise absmax), LoRA on top
#   Adapters (Houlsby et al., 2019 / Pfeiffer et al., 2021)  bottleneck MLP, zero-init up-projection
#   IA3      (Liu et al., 2022)            learned rescaling vectors: keys, values and the FFN inner activation;
#                                          init 1 -> identity; ~0.1% extra parameters
#   Prefix   (Li & Liang, 2021)            learned per-layer key/value prefixes (built in the model code)
#   Prompt   (Lester et al., 2021)         learned soft input tokens (built in the model code)
# ============================================================
class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, self.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A.to(x.dtype)), self.B.to(x.dtype))


class LoRMALinear(nn.Module):
    """LoRMA+ : W = (I_out + (a/r) B A) W0 ;  B in R^{out x r}, A in R^{r x out}. The bias is added after the
    multiplicative transform (it is not part of W0)."""

    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, self.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight.to(x.dtype))
        y = y0 + self.scale * F.linear(F.linear(self.drop(y0), self.A.to(y0.dtype)), self.B.to(y0.dtype))
        return y if self.base.bias is None else y + self.base.bias.to(y.dtype)


NF4_CODE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
                         -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
                         0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
                         0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """Frozen linear layer stored as 4-bit NormalFloat codes (2 per byte) + one fp16 absmax per block of 64.
    Dequantised on the fly in forward (memory saving of QLoRA, without bitsandbytes)."""

    def __init__(self, base, block=64):
        super().__init__()
        W = base.weight.detach().float()
        self.shape, self.block = tuple(W.shape), block
        flat = W.reshape(-1)
        pad = (-flat.numel()) % block
        flat = torch.cat([flat, flat.new_zeros(pad)]).view(-1, block)
        absmax = flat.abs().amax(1, keepdim=True).clamp_min(1e-8)
        code = NF4_CODE.to(flat.device)
        idx = torch.empty(flat.shape, dtype=torch.uint8, device=flat.device)
        for i in range(0, flat.size(0), 4096):                       # chunked argmin keeps memory small
            idx[i:i + 4096] = ((flat[i:i + 4096] / absmax[i:i + 4096]).unsqueeze(-1) - code).abs().argmin(-1)
        idx = idx.view(-1)
        if idx.numel() % 2:
            idx = torch.cat([idx, idx.new_zeros(1)])
        self.register_buffer("packed", (idx[0::2] << 4) | idx[1::2])
        self.register_buffer("absmax", absmax.half())
        self.register_buffer("code", code.clone())
        self.n = W.numel()
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.in_features, self.out_features = self.shape[1], self.shape[0]

    def dequant(self, dtype):
        idx = torch.stack([self.packed >> 4, self.packed & 15], 1).view(-1).long()
        w = self.code[idx][: self.absmax.numel() * self.block].view(-1, self.block) * self.absmax.float()
        return w.view(-1)[: self.n].view(self.shape).to(dtype)

    def forward(self, x):
        return F.linear(x, self.dequant(x.dtype), None if self.bias is None else self.bias.to(x.dtype))


class IA3Linear(nn.Module):
    """IA3: where='out' -> l * W x (keys / values) ; where='in' -> W (l * x) (FFN inner activation, i.e. the input of
    the FFN output projection). l is initialised to 1, so the model starts unchanged."""

    def __init__(self, base, where="out"):
        super().__init__()
        self.base, self.where = base, where
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.l = nn.Parameter(torch.ones(base.out_features if where == "out" else base.in_features))

    def forward(self, x):
        if self.where == "in":
            return self.base(x * self.l.to(x.dtype))
        return self.base(x) * self.l.to(x.dtype)


class Adapter(nn.Module):
    """bottleneck adapter: x + up(act(down(x))); up is zero-initialised so the model starts unchanged"""

    def __init__(self, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        self.drop = nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.drop(self.up(F.gelu(self.down(x))))


class AdapterAfter(nn.Module):
    """Applies an adapter to the output of a sub-layer (Houlsby: after attention and FFN; Pfeiffer: after FFN).
    Works for sub-layers that return a tensor or a tuple (tensor, cache)."""

    def __init__(self, inner, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.inner, self.adapter = inner, Adapter(d, bottleneck, dropout)

    def forward(self, *a, **k):
        out = self.inner(*a, **k)
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)


def replace_linears(root, make, keep):
    """Replace every nn.Linear child c (full name n) under root with make(c) when keep(n) is True."""
    done = []
    for name, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            full = f"{name}.{cname}" if name else cname
            if type(child) is nn.Linear and keep(full):
                setattr(mod, cname, make(child)); done.append(full)
    return done


def n_params(m, trainable=True):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))


def freeze(m):
    for p in m.parameters():
        p.requires_grad = False

In [ ]:
# ============================================================
# 2. ENCODER RUNNERS (from the Phase-6 notebook; Phase 7 uses only ed_encode: the bidirectional encoder embeds each
#    chunk; the decoder is not needed for retrieval and is dropped from the retriever)
#    Original Phase-6 header follows.
# FUSION-IN-DECODER RUNNERS for the pre-trained encoder-decoder (same weights and maths as Phase 1)
#
#   doc -> K chunks (511 tokens + </s>) -> ENCODER on each chunk independently  (bidirectional, 512 window as in
#   pre-training) -> concatenate all chunk states [B, K*512, d] -> DECODER cross-attends to every chunk at once.
#
# Written as functions over the Phase-1 modules (no new weights), adding:
#   * fused SDPA attention everywhere (encoder, decoder self- and cross-attention), dropout passed to the kernel
#   * gradient checkpointing per layer, frozen bottom layers under no_grad
#   * incremental decoding: self-attention KV cache + cross-attention K/V computed ONCE per document and shared by
#     all beams (beams are folded into the query axis, so the 8k-key cross-attention cache is never copied)
#   * learned prefixes for prefix tuning (P hidden vectors per attention, projected by that layer's frozen k / v)
# ============================================================
def unwrap_attn(mod):
    return mod.inner if isinstance(mod, AdapterAfter) else mod


def rel_bias(rpb, q_len, k_len, q_offset=0):
    """T5 relative-position bias [1,H,q_len,k_len] for queries at positions q_offset.. (same bucketing as Phase 1)"""
    dev = rpb.embedding.weight.device
    q = torch.arange(q_offset, q_offset + q_len, device=dev)[:, None]
    k = torch.arange(k_len, device=dev)[None, :]
    b = RelativePositionBias._bucket(k - q, rpb.bidirectional, rpb.num_buckets, rpb.max_distance)
    return rpb.embedding(b).permute(2, 0, 1).unsqueeze(0)


def _heads(mha, x):
    B, L, _ = x.shape
    return x.view(B, L, mha.num_heads, mha.d_head).transpose(1, 2)


def attn_call(mod, xq, xkv, bias, cache=None, static=False):
    """mod: MultiHeadAttention (optionally adapter-wrapped). bias: additive [*, *, Lq, S] or None.
    static=False: self-attention; `cache`=(k,v) of previous steps is extended.
    static=True : cross-attention; `cache`=(k,v) precomputed from the encoder (may serve several beams per source).
    -> (output [B,Lq,d], (k, v))"""
    if isinstance(mod, AdapterAfter):
        out, kv = attn_call(mod.inner, xq, xkv, bias, cache, static)
        return mod.adapter(out), kv
    mha = mod
    B, Lq, D = xq.shape
    H, dh = mha.num_heads, mha.d_head
    q = _heads(mha, mha.q(xq))
    if static and cache is not None:
        k, v = cache
    else:
        k, v = _heads(mha, mha.k(xkv)), _heads(mha, mha.v(xkv))
        if cache is not None:
            k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
    kv = (k, v)
    pre = getattr(mha, "prefix", None)
    if pre is not None:
        P = pre.size(0)
        xp = pre.to(xq.dtype).unsqueeze(0)
        pk, pv = _heads(mha, mha.k(xp)).expand(k.size(0), -1, -1, -1), _heads(mha, mha.v(xp)).expand(k.size(0), -1, -1, -1)
        k, v = torch.cat([pk.to(k.dtype), k], 2), torch.cat([pv.to(v.dtype), v], 2)
        if bias is not None:
            bias = F.pad(bias, (P, 0), value=0.0)
    drop = mha.dropout.p if mha.training else 0.0
    Bk = k.size(0)
    if Bk != B:                                            # several beams per source: fold beams into queries
        assert Lq == 1 and B % Bk == 0
        nb = B // Bk
        qg = q.reshape(Bk, nb, H, dh).permute(0, 2, 1, 3)  # [Bk,H,nb,dh]
        o = F.scaled_dot_product_attention(qg, k, v, attn_mask=None if bias is None else bias.to(q.dtype), dropout_p=drop)
        o = o.permute(0, 2, 1, 3).reshape(B, H, 1, dh)
    else:
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=None if bias is None else bias.to(q.dtype), dropout_p=drop)
    return mha.o(o.transpose(1, 2).reshape(B, Lq, H * dh)), kv


def _first_trainable(layers, x):
    lo = 0
    if torch.is_grad_enabled() and not x.requires_grad:
        while lo < len(layers) and not any(p.requires_grad for p in layers[lo].parameters()):
            lo += 1
    return lo


def ed_encode(ed, ids, am, prompt=None, ckpt=False):
    """ids [M,T] chunks, am [M,T] bool -> (states [M,T',d], mask [M,T']); T' = T + prompt length."""
    enc = ed.encoder
    x = ed.shared_embed(ids)
    if prompt is not None:
        P = prompt.size(0)
        x = torch.cat([prompt.to(x.dtype).unsqueeze(0).expand(x.size(0), -1, -1), x], 1)
        am = torch.cat([am.new_ones(am.size(0), P), am], 1)
    L = x.size(1)
    with torch.no_grad():
        dt = unwrap_attn(enc.layers[0].self_attn).q(x[:1, :1]).dtype      # autocast dtype
    bias = rel_bias(enc.pos_bias, L, L).to(dt).expand(x.size(0), -1, -1, -1).masked_fill(~am[:, None, None, :], float("-inf"))
    x = enc.dropout(x)
    lo = _first_trainable(enc.layers, x) if not bias.requires_grad else 0

    def layer_fn(x, layer):
        h = layer.ln1(x)
        a, _ = attn_call(layer.self_attn, h, h, bias)
        x = x + layer.dropout(a)
        return x + layer.dropout(layer.ff(layer.ln2(x)))

    for i, layer in enumerate(enc.layers):
        if i < lo:
            with torch.no_grad():
                x = layer_fn(x, layer)
        elif ckpt and torch.is_grad_enabled():
            x = grad_ckpt(layer_fn, x, layer, use_reentrant=False)
        else:
            x = layer_fn(x, layer)
    return enc.final_ln(x), am


def ed_decode(ed, y_in, enc_out, enc_mask, cache=None, ckpt=False):
    """y_in [B,Ly] decoder input ids; enc_out [Bs,S,d]; enc_mask [Bs,S] (Bs = B, or B/beams with a cache).
    cache: None (training) or dict(self=[(k,v)|None]*L, cross=[(k,v)]*L, len=int) -> (hidden [B,Ly,d], cache)"""
    dec = ed.decoder
    x = ed.shared_embed(y_in)
    past = 0 if cache is None else cache["len"]
    Lq = x.size(1); S_self = past + Lq
    with torch.no_grad():
        dt = unwrap_attn(dec.layers[0].self_attn).q(x[:1, :1]).dtype
    sb = rel_bias(dec.pos_bias, Lq, S_self, q_offset=past).to(dt)
    if Lq > 1:
        causal = torch.arange(S_self, device=x.device)[None, :] <= (past + torch.arange(Lq, device=x.device))[:, None]
        sb = sb.masked_fill(~causal[None, None], float("-inf"))
    cb = torch.zeros(enc_mask.shape, dtype=dt, device=x.device).masked_fill(~enc_mask, float("-inf"))[:, None, None, :]
    x = dec.dropout(x)
    new_self = []

    def layer_fn(x, layer, sc, cc):
        h = layer.ln1(x)
        a, skv = attn_call(layer.self_attn, h, h, sb, cache=sc)
        x = x + layer.dropout(a)
        h = layer.ln2(x)
        a, ckv = attn_call(layer.cross_attn, h, enc_out, cb, cache=cc, static=True)
        x = x + layer.dropout(a)
        return x + layer.dropout(layer.ff(layer.ln3(x))), skv, ckv

    for i, layer in enumerate(dec.layers):
        sc = None if cache is None else cache["self"][i]
        cc = None if cache is None else cache["cross"][i]
        if ckpt and cache is None and torch.is_grad_enabled():
            x = grad_ckpt(lambda x, layer=layer: layer_fn(x, layer, None, None)[0], x, use_reentrant=False)
        else:
            x, skv, _ = layer_fn(x, layer, sc, cc)
            new_self.append(skv)
    if cache is not None:
        cache = dict(cache, self=new_self, len=S_self)
    return dec.final_ln(x), cache


def ed_cross_cache(ed, enc_out):
    """cross-attention K/V of every decoder layer, computed once per document"""
    out = []
    for layer in ed.decoder.layers:
        mha = unwrap_attn(layer.cross_attn)
        out.append((_heads(mha, mha.k(enc_out)), _heads(mha, mha.v(enc_out))))
    return out


def reorder_dec_cache(cache, idx):
    return dict(cache, self=[None if kv is None else (kv[0].index_select(0, idx), kv[1].index_select(0, idx))
                             for kv in cache["self"]])

In [ ]:
# ============================================================
# 3. THE BIDIRECTIONAL ENCODER AS A CASE ENCODER + ADAPTATION METHODS
#
# Retrieval needs an embedding, not generation, so the retriever keeps only shared_embed + encoder of the Phase-1
# encoder-decoder (the decoder is dropped: fewer deployed parameters). Each chunk = 511 tokens + </s>, the Phase-1
# encoder format. Methods (the Phase-7 list):
#   full | partial (top-k encoder layers) | lorma (LoRMA+) | qlora (NF4 base + LoRA) | ia3 | prefix | prompt | kd
#   (+ lora as an extra, zeroshot = no training)
# ============================================================
D_MODEL = D_ED
CHUNK_BODY = 511
N_HEADS = ED.encoder.layers[0].self_attn.num_heads
D_FF = ED.encoder.layers[0].ff.wi_0.out_features
MODEL_CFG_OVERRIDES = dict(student_layers=max(2, math.ceil(len(ED.encoder.layers) / 2)))


class EncoderOnly(nn.Module):
    def __init__(self, shared_embed, encoder):
        super().__init__()
        self.shared_embed, self.encoder = shared_embed, encoder


ENC_BASE = EncoderOnly(copy.deepcopy(ED.shared_embed), copy.deepcopy(ED.encoder)).cpu().eval()


def finish_chunk(c):
    return np.concatenate([np.asarray(c, dtype=np.int64), [EOS_ID]])


def backbone_embedding(bb):
    return bb.shared_embed


def backbone_encode(bb, ids, am, prompt=None, ckpt=False):
    return ed_encode(bb, ids, am, prompt, ckpt=ckpt)


ED_LINEARS = {"q", "k", "v", "o", "wi_0", "wi_1", "wo"}


def _is_target(name):
    return ".layers." in name and name.split(".")[-1] in ED_LINEARS


def apply_method(e, cfg):
    meth = cfg.method
    freeze(e)
    d = e.shared_embed.embedding_dim
    if meth == "zeroshot":
        return "Phase-1 encoder as is (no training)"
    if meth == "full":
        for p in e.parameters():
            p.requires_grad = True
        return "all encoder weights"
    if meth == "partial":
        for layer in e.encoder.layers[len(e.encoder.layers) - cfg.unfreeze_top_k:]:
            for p in layer.parameters():
                p.requires_grad = True
        for p in e.encoder.final_ln.parameters():
            p.requires_grad = True
        return f"top {cfg.unfreeze_top_k} encoder layers + final norm"
    if meth == "lorma":
        n = replace_linears(e, lambda l: LoRMALinear(l, cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"LoRMA+ r={cfg.r} on {len(n)} linears"
    if meth == "lora":
        n = replace_linears(e, lambda l: LoRALinear(l, cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"LoRA r={cfg.r} on {len(n)} linears"
    if meth == "qlora":
        n = replace_linears(e, lambda l: LoRALinear(NF4Linear(l), cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"QLoRA (NF4 base) r={cfg.r} on {len(n)} linears"
    if meth == "ia3":
        n = 0
        for layer in e.encoder.layers:
            a = layer.self_attn
            a.k, a.v = IA3Linear(a.k, "out"), IA3Linear(a.v, "out")
            layer.ff.wo = IA3Linear(layer.ff.wo, "in")          # scales the gated FFN inner activation
            n += 3
        return f"IA3 on keys, values and FFN inner activation: {n} vectors"
    if meth == "prefix":
        for layer in e.encoder.layers:
            layer.self_attn.prefix = nn.Parameter(torch.randn(cfg.prefix_len, d) * 0.5)
        return f"prefix tuning, {cfg.prefix_len} vectors per encoder self-attention"
    if meth == "prompt":
        return f"prompt tuning, {cfg.prompt_len} soft tokens before every chunk"
    raise ValueError(f"unknown method {meth!r}")


def build_backbone(cfg):
    e = copy.deepcopy(ENC_BASE)
    pre = {id(p) for p in e.parameters()}
    desc = apply_method(e, cfg)
    return e, desc, pre


def make_student_backbone(teacher_bb, cfg):
    """smaller encoder from evenly spaced layers of the fine-tuned teacher (teacher must be a full/partial run)"""
    n = cfg.student_layers
    enc = Encoder(D_MODEL, N_HEADS, D_FF, n)
    src = teacher_bb.encoder
    enc.pos_bias.load_state_dict(src.pos_bias.state_dict())
    enc.final_ln.load_state_dict(src.final_ln.state_dict())
    keep = np.linspace(0, len(src.layers) - 1, n).round().astype(int).tolist()
    for j, i in enumerate(keep):
        enc.layers[j].load_state_dict(src.layers[i].state_dict())
    s = EncoderOnly(copy.deepcopy(teacher_bb.shared_embed), enc)
    print(f"student encoder: {n} layers from teacher layers {keep} | {sum(p.numel() for p in s.parameters())/1e6:.1f}M "
          f"params (teacher {sum(p.numel() for p in teacher_bb.parameters())/1e6:.1f}M)")
    return s


# ---------------- self-test: encoder runner == Phase-1 encoder ----------------
@torch.no_grad()
def check_backbone():
    e = ENC_BASE.to(device).eval()
    docs = [TOK[("train", "c", i)] for i in SPL["train"]["pool_ids"][:50] if len(TOK[("train", "c", i)]) >= 300][:3]
    cuts = [300, 150, 60][:len(docs)]
    T = max(cuts) + 1
    ids = torch.full((len(docs), T), PAD_ID, dtype=torch.long, device=device)
    am = torch.zeros(len(docs), T, dtype=torch.bool, device=device)
    for i, (d, c) in enumerate(zip(docs, cuts)):
        ids[i, :c] = torch.as_tensor(d[:c].astype(np.int64)); ids[i, c] = EOS_ID; am[i, :c + 1] = True
    FAST_ATTN["on"] = False
    ref = e.encoder(e.shared_embed(ids), am.long())
    FAST_ATTN["on"] = True
    new, _ = ed_encode(e, ids, am)
    diff = ((ref - new).abs() * am[..., None]).max().item()
    print(f"encoder runner vs Phase-1 encoder (real positions): max|diff| {diff:.2e}")
    assert diff < 2e-3 * max(1.0, ref.abs().max().item())
    e.cpu()


ED.cpu(); del ED; gc.collect(); torch.cuda.empty_cache()          # only the encoder is kept for retrieval

In [ ]:
# ============================================================
# RETRIEVER: Phase-1 backbone -> chunk vectors -> document representation -> similarity
#
#   document -> 512-token chunks (K spread over the case; K_train sampled at random during training = augmentation)
#            -> backbone (adapted with the chosen method) -> token pooling (attention pooling, zero-init = mean)
#            -> projection d->d (identity init) -> L2-normalised CHUNK vectors
#   score_mode "single": chunk vectors -> attention aggregation (zero-init = mean) -> one vector per case -> cosine
#   score_mode "maxsim": late interaction at chunk level: mean over query chunks of max cosine over candidate chunks
#
# With the identity / zero initialisations, the untrained retriever IS mean-pooled Phase-1 embeddings, so every
# experiment starts from the zero-shot transfer baseline (method="zeroshot" evaluates exactly that).
# Model-specific hooks (defined in the backbone cell): CHUNK_BODY, finish_chunk(), backbone_encode(), D_MODEL.
# ============================================================
class Retriever(nn.Module):
    def __init__(self, bb, cfg):
        super().__init__()
        self.bb, self.cfg = bb, cfg
        d = D_MODEL
        self.tok_w = nn.Parameter(torch.zeros(d))                  # token attention pooling (zero -> mean)
        self.proj = nn.Linear(d, d)
        with torch.no_grad():
            self.proj.weight.copy_(torch.eye(d)); self.proj.bias.zero_()
        self.chunk_w = nn.Parameter(torch.zeros(d))                # chunk aggregation (zero -> mean)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1.0 / cfg.tau)))
        self.prompt = None
        if cfg.method == "prompt":
            emb = backbone_embedding(bb).weight.detach()
            self.prompt = nn.Parameter(emb[torch.randint(1000, 20000, (cfg.prompt_len,))].clone())

    def head_param_ids(self):
        return {id(p) for p in (self.tok_w, self.proj.weight, self.proj.bias, self.chunk_w, self.logit_scale)}

    # ---- chunks -> unit vectors ---------------------------------------------------------------
    def chunk_vec(self, ids, am):
        H, m = backbone_encode(self.bb, ids, am, self.prompt, ckpt=self.cfg.grad_ckpt and self.training)
        m = m.clone()
        if self.prompt is not None:
            m[:, : self.prompt.size(0)] = False                    # pool over the text, not the soft prompt
        H = H.float()
        pool = self.cfg.pool
        if pool == "last":
            last = m.long().cumsum(1).argmax(1)
            v = H[torch.arange(H.size(0), device=H.device), last]
        else:
            logit = (H @ self.tok_w) if pool == "attn" else torch.zeros(m.shape, device=H.device)
            a = torch.softmax(logit.masked_fill(~m, -1e4), dim=1)
            v = (a.unsqueeze(-1) * H).sum(1)
        return F.normalize(self.proj(v), dim=-1)

    # ---- chunk vectors -> per-document representations -------------------------------------------
    def doc_reps(self, V, doc, slot, n_docs):
        K = int(slot.max()) + 1
        mat = V.new_zeros(n_docs, K, V.size(1)).index_put((doc, slot), V)
        msk = torch.zeros(n_docs, K, dtype=torch.bool, device=V.device).index_put((doc, slot), torch.ones_like(doc, dtype=torch.bool))
        a = torch.softmax((mat @ self.chunk_w).masked_fill(~msk, -1e4), dim=1)
        single = F.normalize((a.unsqueeze(-1) * mat).sum(1), dim=-1)
        return single, mat, msk

    def scale(self):
        return self.logit_scale.clamp(max=math.log(100.0)).exp()


def score_reps(q, c, mode, block=64):
    """q, c: (single [n,d], mat [n,K,d], mask [n,K]) -> cosine-type scores [nq, nc] (unscaled)"""
    if mode == "single":
        return q[0] @ c[0].T
    out = []
    cm, cmask = c[1], c[2]
    for s in range(0, q[1].size(0), block):
        qm, qmask = q[1][s:s + block], q[2][s:s + block]
        sim = torch.einsum("bkd,nld->bnkl", qm, cm)
        sim = sim.masked_fill(~cmask[None, :, None, :], -1e4).amax(-1)            # best candidate chunk
        qmf = qmask[:, None, :].to(sim.dtype)
        out.append((sim * qmf).sum(-1) / qmf.sum(-1).clamp_min(1))               # mean over query chunks
    return torch.cat(out, 0)


# ---------------- chunk layout for a list of documents ----------------
def doc_chunk_list(tokens, K, strategy="uniform", rng=None):
    ch = chunk_ids(tokens, CHUNK_BODY, CHUNK_BODY)
    if rng is not None and len(ch) > K:
        ch = [ch[i] for i in sorted(rng.choice(len(ch), K, replace=False))]
    else:
        ch = select_chunks(ch, K, strategy)
    return [finish_chunk(c) for c in ch]


def layout(docs_chunks):
    flat, doc, slot = [], [], []
    for i, chs in enumerate(docs_chunks):
        for j, c in enumerate(chs):
            flat.append(c); doc.append(i); slot.append(j)
    return flat, torch.tensor(doc, device=device), torch.tensor(slot, device=device)


def pad_batch(chunks):
    T = max(max(len(c), 1) for c in chunks)
    ids = torch.full((len(chunks), T), PAD_ID, dtype=torch.long)
    am = torch.zeros(len(chunks), T, dtype=torch.bool)
    for i, c in enumerate(chunks):
        if len(c):
            ids[i, :len(c)] = torch.from_numpy(np.asarray(c, dtype=np.int64)); am[i, :len(c)] = True
        else:
            am[i, 0] = True
    return ids.to(device), am.to(device)


def sub_batches(flat, max_chunks, tok_budget):
    """length-sorted sub-batches bounded by chunk count and token budget"""
    order = sorted(range(len(flat)), key=lambda i: len(flat[i]))
    out, cur = [], []
    for i in order:
        if cur and (len(cur) >= max_chunks or (len(cur) + 1) * max(len(flat[i]), 1) > tok_budget):
            out.append(cur); cur = []
        cur.append(i)
    if cur:
        out.append(cur)
    return out


# ---------------- encoding a whole split (evaluation / caching) ----------------
@torch.no_grad()
def encode_docs(rt, keys, K, strategy="uniform", bs=64, tok_budget=24576):
    """keys: list of TOK keys -> (single [n,d], mat [n,K,d], mask [n,K]) on the GPU (fp16 for the big ones)"""
    rt.eval()
    single, mats, masks = [], [], []
    for s in range(0, len(keys), 256):
        part = keys[s:s + 256]
        flat, doc, slot = layout([doc_chunk_list(TOK[k], K, strategy) for k in part])
        V = torch.empty(len(flat), D_MODEL, device=device)
        for sb in sub_batches(flat, bs, tok_budget):
            ids, am = pad_batch([flat[i] for i in sb])
            with amp_ctx():
                V[torch.tensor(sb, device=device)] = rt.chunk_vec(ids, am).float()
        sg, mt, mk = rt.doc_reps(V, doc, slot, len(part))
        single.append(sg.half()); mats.append(F.pad(mt, (0, 0, 0, K - mt.size(1))).half())
        masks.append(F.pad(mk.float(), (0, K - mk.size(1))).bool())
    return torch.cat(single), torch.cat(mats), torch.cat(masks)


@torch.no_grad()
def split_scores(rt, split, cfg, reps=None):
    """-> numpy [nq, nc] scores for the split, plus the query / pool representations"""
    if reps is None:
        t0 = time.time()
        q = encode_docs(rt, [(split, "q", i) for i in SPL[split]["qids"]], cfg.k_eval, cfg.chunk_strategy)
        c = encode_docs(rt, [(split, "c", i) for i in SPL[split]["pool_ids"]], cfg.k_eval, cfg.chunk_strategy)
        enc_s = time.time() - t0
    else:
        (q, c), enc_s = reps, 0.0
    S = score_reps(tuple(x.float() if x.dtype == torch.float16 else x for x in q),
                   tuple(x.float() if x.dtype == torch.float16 else x for x in c), cfg.score_mode)
    return S.float().cpu().numpy(), (q, c), enc_s

In [ ]:
# ============================================================
# CONTRASTIVE TRAINING with GradCache (Gao et al., 2021)
#
# One step = B training queries, each with 1 relevant candidate + n_hard hard negatives; every other candidate in the
# batch is an in-batch negative (candidates relevant to the query are masked, never used as negatives).
# Long cases make each "example" K_train x 512 tokens, so a 40-document batch is ~160 chunk forwards. GradCache keeps
# the large contrastive batch on a T4:
#   1) encode every chunk WITHOUT a graph (RNG state saved per sub-batch)      -> chunk vectors V
#   2) loss(V) with the document aggregation; backward to V                  -> dL/dV (and grads of the aggregator)
#   3) re-encode each sub-batch WITH a graph under the same RNG state and back-propagate dL/dV through it
# The gradient is exactly that of the full batch, with activation memory of one sub-batch.
#
# Objectives: "infonce" (softmax over the batch, Oord et al. 2018) | "pairwise" (logistic loss on every
# positive-negative pair, RankNet-style). KD (Hinton et al., 2015): listwise KL between the teacher's and the student's
# score distributions over the same batch + cosine alignment of student and teacher case vectors.
# ============================================================
def hard_negative_table(S, split, depth):
    """top-`depth` non-relevant, allowed candidates per query from a score matrix (BM25, teacher or the model)"""
    S = np.where(valid_mask(split), S, -np.inf)
    pids, rel = SPL[split]["pool_ids"], SPL[split]["rel"]
    order = np.argsort(-S, axis=1, kind="stable")[:, : depth + 50]
    table = {}
    for i, q in enumerate(SPL[split]["qids"]):
        table[q] = [pids[j] for j in order[i] if np.isfinite(S[i, j]) and pids[j] not in rel[q]][:depth]
    return table


def make_batch(qids, hn, cfg, rng):
    rel = SPL["train"]["rel"]
    allowed = SPL["train"].get("allowed")
    pool_ids = SPL["train"]["pool_ids"]
    cands, pos_col = [], []
    for q in qids:
        pos = str(rng.choice(sorted(rel[q])))
        pos_col.append(len(cands)); cands.append(pos)
        neg_src = hn.get(q) if hn else None
        if not neg_src:
            src = allowed[q] if allowed else pool_ids
            neg_src = [c for c in rng.choice(src, min(len(src), 4 * cfg.n_hard + 4), replace=False)
                       if c not in rel[q] and c != q]
        k = min(cfg.n_hard, len(neg_src))
        cands += [str(c) for c in rng.choice(neg_src[: cfg.hn_depth], k, replace=False)] if k else []
    B, Nc = len(qids), len(cands)
    invalid = torch.zeros(B, Nc, dtype=torch.bool)
    for i, q in enumerate(qids):
        ok = set(allowed[q]) if allowed else None
        for j, c in enumerate(cands):
            if j != pos_col[i] and (c in rel[q] or c == q or (ok is not None and c not in ok)):
                invalid[i, j] = True
    return cands, torch.tensor(pos_col), invalid


def batch_loss(rt, V, doc, slot, B, pos_col, invalid, cfg, teacher_T=None, teacher_rep=None):
    n_docs = int(doc.max()) + 1
    single, mat, msk = rt.doc_reps(V, doc, slot, n_docs)
    q = (single[:B], mat[:B], msk[:B]); c = (single[B:], mat[B:], msk[B:])
    S = score_reps(q, c, cfg.score_mode) * rt.scale()
    inv = invalid.to(S.device)
    S = S.masked_fill(inv, -1e4)
    pc = pos_col.to(S.device)
    if cfg.objective == "infonce":
        loss = F.cross_entropy(S, pc)
    elif cfg.objective == "pairwise":
        sp_ = S.gather(1, pc[:, None])
        neg = ~inv; neg[torch.arange(B), pc] = False
        loss = F.softplus(S - sp_)[neg].mean()
    else:
        raise ValueError(cfg.objective)
    if teacher_T is not None:
        T = teacher_T.to(S.device).masked_fill(inv, -1e4)
        t = cfg.kd_temp
        kl = F.kl_div(F.log_softmax(S / t, -1), F.log_softmax(T / t, -1), log_target=True, reduction="batchmean") * t * t
        loss = (1 - cfg.kd_alpha) * loss + cfg.kd_alpha * kl
    if teacher_rep is not None and cfg.kd_rep > 0:
        loss = loss + cfg.kd_rep * (1 - (single * teacher_rep.to(single.device)).sum(-1)).mean()
    return loss


def gradcache_step(rt, flat, doc, slot, loss_fn, scaler, cfg):
    sbs = sub_batches(flat, cfg.enc_bs, cfg.enc_tok_budget)
    V = torch.empty(len(flat), D_MODEL, device=device)
    states = []
    with torch.no_grad():
        for sb in sbs:
            states.append(get_rng())
            ids, am = pad_batch([flat[i] for i in sb])
            with amp_ctx():
                V[torch.tensor(sb, device=device)] = rt.chunk_vec(ids, am).float()
    V.requires_grad_(True)
    loss = loss_fn(V, doc, slot)
    scaler.scale(loss).backward()
    G = V.grad.detach()
    if not torch.isfinite(G).all():
        return float("nan")
    for sb, st in zip(sbs, states):
        set_rng(st)
        ids, am = pad_batch([flat[i] for i in sb])
        idx = torch.tensor(sb, device=device)
        with amp_ctx():
            v = rt.chunk_vec(ids, am).float()
        v.backward(G[idx])
    return float(loss)


def train_retriever(rt, cfg, tag, teacher_cache=None):
    rng = np.random.RandomState(cfg.seed)
    pre, head = rt.pretrained_ids, rt.head_param_ids()
    groups = []
    for name, sel, lr in (("head", lambda p: id(p) in head, cfg.lr_head),
                          ("new", lambda p: id(p) not in head and id(p) not in pre, cfg.lr),
                          ("backbone", lambda p: id(p) in pre, cfg.lr_backbone)):
        ps = [p for p in rt.parameters() if p.requires_grad and sel(p)]
        if ps:
            groups.append(dict(params=ps, lr=lr, name=name))
    opt = torch.optim.AdamW(groups, weight_decay=cfg.wd)
    train_q = [q for q in SPL["train"]["qids"] if SPL["train"]["rel"][q]][: cfg.max_train_q or None]
    steps_ep = math.ceil(len(train_q) / cfg.batch_q)
    total = steps_ep * cfg.epochs; warm = max(1, int(cfg.warmup * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.05, 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)

    if cfg.hn_source == "teacher":
        assert teacher_cache is not None, "hn_source='teacher' needs a teacher"
        hn = hard_negative_table(teacher_cache["S"], "train", cfg.hn_depth)
    elif cfg.hn_source == "bm25":
        hn = hard_negative_table(bm25_scores("train"), "train", cfg.hn_depth)
    elif cfg.hn_source == "dense":                              # the model's own (zero-shot at start) ranking
        hn = hard_negative_table(split_scores(rt, "train", cfg)[0], "train", cfg.hn_depth)
    else:
        hn = None
    t_idx = None
    if teacher_cache is not None:
        t_idx = {("q", k): i for i, k in enumerate(SPL["train"]["qids"])}
        t_idx.update({("c", k): i for i, k in enumerate(SPL["train"]["pool_ids"])})

    best, bad = -1.0, 0
    ck_path = os.path.join(local_ck, f"{tag}.pt")
    print(f"[{tag}] {len(train_q)} train queries | {steps_ep} steps/epoch x {cfg.epochs} | batch {cfg.batch_q} queries "
          f"x (1 pos + {cfg.n_hard} hard neg, hn={cfg.hn_source}) | K_train {cfg.k_train} chunks | "
          + " ".join(f"{g['name']} {sum(p.numel() for p in g['params'])/1e6:.2f}M" for g in groups))
    for ep in range(cfg.epochs):
        rt.train(); t0, run, n = time.time(), 0.0, 0
        perm = rng.permutation(len(train_q))
        for st in range(steps_ep):
            qids = [train_q[i] for i in perm[st * cfg.batch_q:(st + 1) * cfg.batch_q]]
            if not qids:
                continue
            cands, pos_col, invalid = make_batch(qids, hn, cfg, rng)
            keys = [("train", "q", q) for q in qids] + [("train", "c", c) for c in cands]
            flat, doc, slot = layout([doc_chunk_list(TOK[k], cfg.k_train, cfg.chunk_strategy, rng) for k in keys])
            tT = tR = None
            if teacher_cache is not None:
                qi = torch.tensor([t_idx[("q", q)] for q in qids]); ci = torch.tensor([t_idx[("c", c)] for c in cands])
                tq = tuple(x[qi.to(x.device)].float() for x in teacher_cache["q"])
                tc = tuple(x[ci.to(x.device)].float() for x in teacher_cache["c"])
                tT = score_reps(tq, tc, teacher_cache["mode"]) * teacher_cache["scale"]
                tR = torch.cat([tq[0], tc[0]], 0)
            fn = lambda V, d, s: batch_loss(rt, V, d, s, len(qids), pos_col, invalid, cfg, tT, tR)
            loss = gradcache_step(rt, flat, doc, slot, fn, scaler, cfg)
            if not math.isfinite(loss):                         # fp16 overflow in dL/dV: skip, halve the loss scale
                print(f"[{tag}] non-finite step skipped"); opt.zero_grad(set_to_none=True)
                if scaler.is_enabled():
                    scaler.update(scaler.get_scale() / 2)
                continue
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], cfg.clip)
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
            run += loss; n += 1
            if (st + 1) % cfg.log_every == 0:
                print(f"[{tag}] ep {ep+1} step {st+1}/{steps_ep} loss {run/n:.4f} scale {float(rt.scale()):.1f} "
                      f"({time.time()-t0:.0f}s)")
        S, _, _ = split_scores(rt, "dev", cfg)
        m = full_metrics(S, "dev")
        print(f"[{tag}] ep {ep+1} train loss {run/max(n,1):.4f} | DEV {fmt_m(m)} ({time.time()-t0:.0f}s)")
        if m["f1"] > best:
            best, bad = m["f1"], 0
            save_trainable(rt, ck_path, {"cfg": vars(cfg), "epoch": ep + 1, "dev": m})
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"[{tag}] early stop"); break
        if cfg.hn_refresh and ep + 1 < cfg.epochs:              # ANCE-style: re-mine with the current model
            hn = hard_negative_table(split_scores(rt, "train", cfg)[0], "train", cfg.hn_depth)
            print(f"[{tag}] hard negatives re-mined with the current model")
    load_trainable(rt, ck_path)
    shutil.copy(ck_path, os.path.join(p7_dir, f"ckpt_{tag}.pt"))
    return best


@torch.no_grad()
def build_teacher_cache(teacher, tcfg):
    """teacher representations of every train query / candidate + the full train score matrix (for KD + hard negatives)"""
    S, (q, c), _ = split_scores(teacher, "train", tcfg)
    return dict(S=S, q=q, c=c, mode=tcfg.score_mode, scale=float(teacher.scale()))

In [ ]:
# ============================================================
# CONFIG + EXPERIMENT RUNNER
# ============================================================
BASE_CFG = dict(
    method="lorma",
    # document representation
    k_train=4, k_eval=8, chunk_strategy="uniform", pool="attn", score_mode="single",
    # objective / negatives
    objective="infonce", tau=0.05, batch_q=8, n_hard=3, hn_source="bm25", hn_depth=50, hn_refresh=False,
    # adaptation hyper-parameters
    r=16, alpha=32, lora_dropout=0.05, prefix_len=16, prompt_len=20, unfreeze_top_k=3,
    # optimisation
    lr=5e-4, lr_head=1e-3, lr_backbone=2e-5, wd=0.01, epochs=3, patience=2, warmup=0.06, clip=1.0,
    grad_ckpt=True, enc_bs=16, enc_tok_budget=8192, seed=42, log_every=25, max_train_q=None,
    # knowledge distillation
    teacher_ckpt=None, kd_alpha=0.5, kd_temp=1.0, kd_rep=0.5, student_layers=None,
)
METHOD_DEFAULTS = {
    "zeroshot": dict(),
    "full":     dict(lr_backbone=2e-5),
    "partial":  dict(lr_backbone=5e-5),
    "lorma":    dict(lr=5e-4),
    "qlora":    dict(lr=5e-4),
    "lora":     dict(lr=5e-4),
    "ia3":      dict(lr=3e-3),
    "prefix":   dict(lr=3e-3),
    "prompt":   dict(lr=3e-3),
    "kd":       dict(lr_backbone=5e-5, hn_source="teacher"),
}


def make_cfg(**over):
    meth = over.get("method", BASE_CFG["method"])
    return SimpleNamespace(**{**BASE_CFG, **MODEL_CFG_OVERRIDES, **METHOD_DEFAULTS.get(meth, {}), **over})


RESULTS = {}
_res_path = os.path.join(p7_dir, "results.json")
if os.path.exists(_res_path):
    RESULTS = json.load(open(_res_path))


def build_retriever(cfg, backbone=None):
    """backbone=None: fresh copy of the Phase-1 model with cfg.method applied; else a (student) backbone."""
    if backbone is None:
        bb, desc, pre = build_backbone(cfg)          # pre = ids of the Phase-1 weights, collected BEFORE injection
    else:
        bb, desc = backbone, "student backbone (all weights trainable)"
        pre = {id(p) for p in bb.parameters()}
    rt = Retriever(bb, cfg).to(device)
    rt.pretrained_ids = pre                          # -> lr_backbone; PEFT parameters -> lr; head -> lr_head
    rt.method_desc = desc
    return rt


def _from_ckpt(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = SimpleNamespace(**ck["cfg"])
    if cfg.method == "kd":
        tch, _ = _from_ckpt(cfg.teacher_ckpt)
        rt = build_retriever(cfg, backbone=make_student_backbone(tch.bb, cfg)); del tch
    else:
        rt = build_retriever(cfg)
    load_trainable(rt, path)
    return rt, cfg


def evaluate_and_submit(rt, cfg, name, submit=True):
    S_dev, _, enc_dev = split_scores(rt, "dev", cfg)
    m = full_metrics(S_dev, "dev")
    (w, f1h), _ = tune_fusion(S_dev, "dev")
    nq = len(SPL["dev"]["qids"]); npool = len(SPL["dev"]["pool_ids"])
    print(f"[{name}] DEV dense : {fmt_m(m)}")
    print(f"[{name}] DEV hybrid: F1@7 {f1h:.4f} with w_dense={w:.1f} (BM25 alone = w 0.0)")
    RESULTS[name] = dict(dev=m, dev_hybrid=dict(f1=f1h, w=w), method=cfg.method, desc=rt.method_desc,
                         trainable=sum(p.numel() for p in rt.parameters() if p.requires_grad),
                         deployed=sum(p.numel() for p in rt.parameters()),
                         enc_sec_per_doc=enc_dev / max(nq + npool, 1),
                         cfg={k: v for k, v in vars(cfg).items() if isinstance(v, (int, float, str, bool, type(None)))})
    json.dump(RESULTS, open(_res_path, "w"), indent=1)
    if submit:
        S_test, _, _ = split_scores(rt, "test", cfg)
        write_submission(S_test, name)
        if w < 1.0:
            write_submission(w * zrow(S_test) + (1 - w) * zrow(bm25_scores("test")), name + "_hybrid")
    return m


def run_experiment(name, submit=True, **over):
    cfg = make_cfg(**over)
    set_seed(cfg.seed)
    tcache = None
    if cfg.method == "kd":
        assert cfg.teacher_ckpt and os.path.exists(cfg.teacher_ckpt), "kd needs teacher_ckpt (e.g. ckpt of the full run)"
        teacher, tcfg = _from_ckpt(cfg.teacher_ckpt)
        for k in ("pool", "score_mode", "k_eval"):
            setattr(cfg, k, getattr(tcfg, k))
        print("caching teacher representations of the train split ...")
        tcache = build_teacher_cache(teacher, tcfg)
        rt = build_retriever(cfg, backbone=make_student_backbone(teacher.bb, cfg))
        rt.load_state_dict({k: v for k, v in teacher.state_dict().items() if not k.startswith("bb.")}, strict=False)
        del teacher; gc.collect(); torch.cuda.empty_cache()
    else:
        rt = build_retriever(cfg)
    print(f"\n=== {name} | {rt.method_desc} | trainable {sum(p.numel() for p in rt.parameters() if p.requires_grad)/1e6:.2f}M"
          f" / {sum(p.numel() for p in rt.parameters())/1e6:.1f}M ===")
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    if cfg.method != "zeroshot":
        train_retriever(rt, cfg, name, teacher_cache=tcache)
    train_min = (time.time() - t0) / 60
    peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0.0
    del tcache
    m = evaluate_and_submit(rt, cfg, name, submit)
    RESULTS[name].update(train_min=train_min, peak_gb=peak)
    json.dump(RESULTS, open(_res_path, "w"), indent=1)
    del rt; gc.collect(); torch.cuda.empty_cache()
    return m


def bm25_reference(submit=False):
    """lexical reference point (not a Phase-1-model system; submit only if you want it on the leaderboard)"""
    m = full_metrics(bm25_scores("dev"), "dev")
    print("BM25 (BPE unigrams + bigrams) DEV:", fmt_m(m))
    RESULTS["BM25_reference"] = dict(dev=m, method="bm25", desc="BM25 on BPE ids", trainable=0, deployed=0)
    json.dump(RESULTS, open(_res_path, "w"), indent=1)
    if submit:
        write_submission(bm25_scores("test"), "BM25_reference")
    return m


def show_results():
    print(f"\n{'experiment':24s} {'method':9s} {'F1@7':>7s} {'P@7':>7s} {'R@7':>7s} {'R@100':>6s} {'MRR':>6s} "
          f"{'hybrid':>7s} {'train M':>8s} {'model M':>8s} {'ms/doc':>7s} {'min':>5s} {'GB':>5s}")
    for n, r in RESULTS.items():
        d = r["dev"]
        print(f"{n:24s} {r['method']:9s} {d['f1']:7.4f} {d['p']:7.4f} {d['r']:7.4f} {d.get('r100', 0):6.3f} "
              f"{d.get('mrr', 0):6.3f} {r.get('dev_hybrid', {}).get('f1', 0):7.4f} {r['trainable']/1e6:8.2f} "
              f"{r['deployed']/1e6:8.1f} {1000 * r.get('enc_sec_per_doc', 0):7.1f} {r.get('train_min', 0):5.0f} "
              f"{r.get('peak_gb', 0):5.1f}")

In [ ]:
# ============================================================
# SELF-TESTS (a minute): run before training
#   1) the backbone patch reproduces the Phase-1 model
#   2) LoRMA / LoRA / IA3 leave the retriever unchanged at init (QLoRA: 4-bit base stays close)
#   3) GradCache gives the SAME gradients as ordinary back-propagation through the whole batch (dropout on)
# ============================================================
@torch.no_grad()
def check_peft_identity():
    keys = [("train", "c", i) for i in SPL["train"]["pool_ids"][:6]]
    ref = encode_docs(build_retriever(make_cfg(method="zeroshot")), keys, 2)[0].float()
    for meth in ("lorma", "lora", "ia3", "qlora"):
        rt = build_retriever(make_cfg(method=meth))
        out = encode_docs(rt, keys, 2)[0].float()
        cos = (out * ref).sum(-1).min().item()
        print(f"  {meth:6s} at init: min cosine to zero-shot case vectors {cos:.5f}" + ("  (4-bit base)" if meth == "qlora" else ""))
        del rt
    torch.cuda.empty_cache()


def check_gradcache():
    global AMP_DTYPE
    saved, AMP_DTYPE = AMP_DTYPE, None                        # fp32 for a tight comparison
    try:
        cfg = make_cfg(method="lorma", batch_q=2, n_hard=1, k_train=2, enc_bs=2)
        rt = build_retriever(cfg); rt.train()
        rng = np.random.RandomState(0)
        qids = [q for q in SPL["train"]["qids"] if SPL["train"]["rel"][q]][:2]
        cands, pos, inv = make_batch(qids, None, cfg, rng)
        keys = [("train", "q", q) for q in qids] + [("train", "c", c) for c in cands]
        flat, doc, slot = layout([doc_chunk_list(TOK[k], cfg.k_train, cfg.chunk_strategy, rng) for k in keys])
        for i in range(len(flat)):
            flat[i] = flat[i][:128]                           # short chunks keep the test fast
        fn = lambda V, d, s: batch_loss(rt, V, d, s, len(qids), pos, inv, cfg)
        params = [(n, p) for n, p in rt.named_parameters() if p.requires_grad]
        st0 = get_rng()
        l1 = gradcache_step(rt, flat, doc, slot, fn, torch.amp.GradScaler("cuda", enabled=False), cfg)
        g1 = {n: p.grad.detach().clone() for n, p in params if p.grad is not None}
        rt.zero_grad(set_to_none=True)
        set_rng(st0)
        V = torch.zeros(len(flat), D_MODEL, device=device)
        for sb in sub_batches(flat, cfg.enc_bs, cfg.enc_tok_budget):
            ids, am = pad_batch([flat[i] for i in sb])
            V = V.index_put((torch.tensor(sb, device=device),), rt.chunk_vec(ids, am).float())
        l2 = fn(V, doc, slot); l2.backward()
        worst = 0.0
        for n, p in params:
            if p.grad is None or n not in g1:
                continue
            worst = max(worst, ((g1[n] - p.grad).abs().max() / (p.grad.abs().max() + 1e-8)).item())
        print(f"GradCache vs direct backprop: loss {l1:.5f} vs {float(l2):.5f} | worst relative grad diff {worst:.2e}")
        assert abs(l1 - float(l2)) < 1e-4 and worst < 1e-2, "GradCache gradients differ from direct back-propagation"
        del rt; torch.cuda.empty_cache()
    finally:
        AMP_DTYPE = saved


check_backbone()
check_peft_identity()
check_gradcache()

encoder runner vs Phase-1 encoder (real positions): max|diff| 1.67e-06
  lorma  at init: min cosine to zero-shot case vectors 0.99994
  lora   at init: min cosine to zero-shot case vectors 0.99994
  ia3    at init: min cosine to zero-shot case vectors 0.99994
  qlora  at init: min cosine to zero-shot case vectors 0.99874  (4-bit base)


/tmp/ipykernel_1330/2604840094.py:102: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  return float(loss)


GradCache vs direct backprop: loss 2.82437 vs 2.82437 | worst relative grad diff 2.08e-07


In [ ]:
# ============================================================
# 5. RUN  (dev Micro-F1@7 is printed after every epoch; every run writes submission_<name>/submission.zip, plus a
#          _hybrid variant (dense + BM25 fusion, weight tuned on dev) when fusion helps)
# ============================================================
# (0) Reference points (minutes, no training): lexical BM25 and the zero-shot Phase-1 encoder
bm25_reference()
run_experiment("D0_zeroshot", method="zeroshot", submit=False)
run_experiment("D_smoke", method="lorma", max_train_q=32, epochs=1, submit=False)   # whole pipeline, a few minutes

# (1) Six substantially different techniques from the Phase-7 list (LoRMA compulsory, KD mandatory).
#     Same representation (8 chunks, attention pooling, single vector), same objective and negatives,
#     so the differences come from the adaptation technique.
# run_experiment("D1_full",    method="full", batch_q=8)                 # strongest; the KD teacher
# run_experiment("D2_lorma",   method="lorma")                           # compulsory
# run_experiment("D3_ia3",     method="ia3")
# run_experiment("D4_prefix",  method="prefix")
# run_experiment("D5_partial", method="partial", unfreeze_top_k=3)
# run_experiment("D6_kd",      method="kd", teacher_ckpt=os.path.join(p7_dir, "ckpt_D1_full.pt"))   # student = half the encoder layers

# (2) Extras: other techniques, and representation / objective / negative ablations (not separate techniques)
# run_experiment("D7_qlora",    method="qlora")
# run_experiment("D8_prompt",   method="prompt")
# run_experiment("D9_maxsim",   method="lorma", score_mode="maxsim")     # chunk-level late interaction
# run_experiment("D10_mean",    method="lorma", pool="mean")             # plain mean pooling
# run_experiment("D11_pair",    method="lorma", objective="pairwise")
# run_experiment("D12_randneg", method="lorma", hn_source="none")        # in-batch + random negatives only
# run_experiment("D13_ance",    method="lorma", hn_source="dense", hn_refresh=True)
# run_experiment("D14_head",    method="lorma", chunk_strategy="head")

show_results()

BM25 (BPE unigrams + bigrams) DEV: F1@7 0.2034 (P 0.2215 R 0.1881) | R@50 0.471 R@100 0.581 MRR 0.587

=== D0_zeroshot | Phase-1 encoder as is (no training) | trainable 0.26M / 54.8M ===
[D0_zeroshot] DEV dense : F1@7 0.2090 (P 0.2276 R 0.1932) | R@50 0.567 R@100 0.659 MRR 0.536
[D0_zeroshot] DEV hybrid: F1@7 0.2590 with w_dense=0.7 (BM25 alone = w 0.0)

=== D_smoke | LoRMA+ r=16 on 63 linears | trainable 2.18M / 56.7M ===
[D_smoke] 32 train queries | 4 steps/epoch x 1 | batch 8 queries x (1 pos + 3 hard neg, hn=bm25) | K_train 4 chunks | head 0.26M new 1.92M


/tmp/ipykernel_1330/2604840094.py:168: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)


[D_smoke] ep 1 train loss 2.7617 | DEV F1@7 0.2235 (P 0.2433 R 0.2066) | R@50 0.595 R@100 0.673 MRR 0.554 (75s)
[D_smoke] DEV dense : F1@7 0.2235 (P 0.2433 R 0.2066) | R@50 0.595 R@100 0.673 MRR 0.554
[D_smoke] DEV hybrid: F1@7 0.2679 with w_dense=0.7 (BM25 alone = w 0.0)

experiment               method       F1@7     P@7     R@7  R@100    MRR  hybrid  train M  model M  ms/doc   min    GB
BM25_reference           bm25       0.2034  0.2215  0.1881  0.581  0.587  0.0000     0.00      0.0     0.0     0   0.0
D0_zeroshot              zeroshot   0.2090  0.2276  0.1932  0.659  0.536  0.2590     0.26     54.8    30.4     0   0.2
D_smoke                  lorma      0.2235  0.2433  0.2066  0.673  0.554  0.2679     2.18     56.7    44.1     2   1.3


In [ ]:
run_experiment("D1_full",    method="full", batch_q=8)                 # strongest; the KD teacher



=== D1_full | all encoder weights | trainable 54.80M / 54.8M ===
[D1_full] 817 train queries | 103 steps/epoch x 3 | batch 8 queries x (1 pos + 3 hard neg, hn=bm25) | K_train 4 chunks | head 0.26M backbone 54.54M


/tmp/ipykernel_1330/2604840094.py:168: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)


[D1_full] ep 1 step 25/103 loss 2.6410 scale 19.9 (120s)
[D1_full] ep 1 step 50/103 loss 2.2225 scale 19.8 (239s)
[D1_full] ep 1 step 75/103 loss 2.1268 scale 19.6 (358s)
[D1_full] ep 1 step 100/103 loss 2.0625 scale 19.5 (477s)
[D1_full] ep 1 train loss 2.0339 | DEV F1@7 0.2902 (P 0.3160 R 0.2682) | R@50 0.668 R@100 0.756 MRR 0.670 (526s)
[D1_full] ep 2 step 25/103 loss 1.8236 scale 19.4 (118s)
[D1_full] ep 2 step 50/103 loss 1.8906 scale 19.3 (238s)
[D1_full] ep 2 step 75/103 loss 1.8431 scale 19.3 (356s)
[D1_full] ep 2 step 100/103 loss 1.7847 scale 19.3 (475s)
[D1_full] ep 2 train loss 1.7708 | DEV F1@7 0.3046 (P 0.3317 R 0.2816) | R@50 0.690 R@100 0.747 MRR 0.703 (523s)
[D1_full] ep 3 step 25/103 loss 1.7109 scale 19.3 (119s)
[D1_full] ep 3 step 50/103 loss 1.7139 scale 19.3 (237s)
[D1_full] ep 3 step 75/103 loss 1.6024 scale 19.3 (357s)
[D1_full] ep 3 step 100/103 loss 1.6473 scale 19.2 (475s)
[D1_full] ep 3 train loss 1.6501 | DEV F1@7 0.3091 (P 0.3366 R 0.2857) | R@50 0.698 R@1

{'f1': 0.3090605892162312,
 'p': 0.3365617433414044,
 'r': 0.2857142857142857,
 'r50': 0.6977434344343028,
 'r100': 0.750093298476892,
 'mrr': 0.7102829918619394}

In [ ]:
run_experiment("D2_lorma",   method="lorma")                           # compulsory



=== D2_lorma | LoRMA+ r=16 on 63 linears | trainable 2.18M / 56.7M ===
[D2_lorma] 817 train queries | 103 steps/epoch x 3 | batch 8 queries x (1 pos + 3 hard neg, hn=bm25) | K_train 4 chunks | head 0.26M new 1.92M


/tmp/ipykernel_1330/2604840094.py:168: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)


[D2_lorma] ep 1 step 25/103 loss 2.6220 scale 19.9 (145s)
[D2_lorma] ep 1 step 50/103 loss 2.1992 scale 19.8 (289s)
[D2_lorma] ep 1 step 75/103 loss 2.1340 scale 19.6 (434s)
[D2_lorma] ep 1 step 100/103 loss 2.0946 scale 19.5 (579s)
[D2_lorma] ep 1 train loss 2.0635 | DEV F1@7 0.2902 (P 0.3160 R 0.2682) | R@50 0.689 R@100 0.762 MRR 0.682 (642s)
[D2_lorma] ep 2 step 25/103 loss 1.8844 scale 19.4 (144s)
[D2_lorma] ep 2 step 50/103 loss 1.9290 scale 19.3 (289s)
[D2_lorma] ep 2 step 75/103 loss 1.8756 scale 19.2 (432s)
[D2_lorma] ep 2 step 100/103 loss 1.8064 scale 19.2 (577s)
[D2_lorma] ep 2 train loss 1.7933 | DEV F1@7 0.3091 (P 0.3366 R 0.2857) | R@50 0.692 R@100 0.762 MRR 0.701 (640s)
[D2_lorma] ep 3 step 25/103 loss 1.7220 scale 19.2 (145s)
[D2_lorma] ep 3 step 50/103 loss 1.7259 scale 19.2 (289s)
[D2_lorma] ep 3 step 75/103 loss 1.5991 scale 19.2 (434s)
[D2_lorma] ep 3 step 100/103 loss 1.6402 scale 19.2 (578s)
[D2_lorma] ep 3 train loss 1.6398 | DEV F1@7 0.3102 (P 0.3378 R 0.2867) |

{'f1': 0.3101723179544191,
 'p': 0.33777239709443097,
 'r': 0.2867420349434738,
 'r50': 0.7005384739121855,
 'r100': 0.7650489565737668,
 'mrr': 0.7060314931114375}

In [ ]:
run_experiment("D3_ia3",     method="ia3")



=== D3_ia3 | IA3 on keys, values and FFN inner activation: 27 vectors | trainable 0.29M / 54.8M ===
[D3_ia3] 817 train queries | 103 steps/epoch x 3 | batch 8 queries x (1 pos + 3 hard neg, hn=bm25) | K_train 4 chunks | head 0.26M new 0.03M


/tmp/ipykernel_1330/2604840094.py:168: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)


[D3_ia3] ep 1 step 25/103 loss 2.6641 scale 19.9 (113s)
[D3_ia3] ep 1 step 50/103 loss 2.2434 scale 19.8 (226s)
[D3_ia3] ep 1 step 75/103 loss 2.1518 scale 19.5 (338s)
[D3_ia3] ep 1 step 100/103 loss 2.0828 scale 19.4 (451s)
[D3_ia3] ep 1 train loss 2.0535 | DEV F1@7 0.2979 (P 0.3245 R 0.2754) | R@50 0.683 R@100 0.755 MRR 0.658 (503s)
[D3_ia3] ep 2 step 25/103 loss 1.8865 scale 19.2 (112s)
[D3_ia3] ep 2 step 50/103 loss 1.9424 scale 19.1 (225s)
[D3_ia3] ep 2 step 75/103 loss 1.8982 scale 19.1 (338s)
[D3_ia3] ep 2 step 100/103 loss 1.8402 scale 19.1 (451s)
[D3_ia3] ep 2 train loss 1.8269 | DEV F1@7 0.2991 (P 0.3257 R 0.2765) | R@50 0.698 R@100 0.756 MRR 0.693 (503s)
[D3_ia3] ep 3 step 25/103 loss 1.7674 scale 19.0 (113s)
[D3_ia3] ep 3 step 50/103 loss 1.7823 scale 19.0 (226s)
[D3_ia3] ep 3 step 75/103 loss 1.6607 scale 19.0 (339s)
[D3_ia3] ep 3 step 100/103 loss 1.7021 scale 19.0 (451s)
[D3_ia3] ep 3 train loss 1.7071 | DEV F1@7 0.2968 (P 0.3232 R 0.2744) | R@50 0.703 R@100 0.755 MRR 0.

{'f1': 0.29905503057254035,
 'p': 0.32566585956416466,
 'r': 0.27646454265159304,
 'r50': 0.6980162218459027,
 'r100': 0.755637751362907,
 'mrr': 0.6933319096082287}

In [ ]:
run_experiment("D4_prefix",  method="prefix")


=== D4_prefix | prefix tuning, 16 vectors per encoder self-attention | trainable 0.34M / 54.9M ===
[D4_prefix] 817 train queries | 103 steps/epoch x 3 | batch 8 queries x (1 pos + 3 hard neg, hn=bm25) | K_train 4 chunks | head 0.26M new 0.07M


/tmp/ipykernel_1330/2604840094.py:168: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)


[D4_prefix] ep 1 step 25/103 loss 2.6795 scale 19.9 (120s)
[D4_prefix] ep 1 step 50/103 loss 2.2420 scale 19.8 (239s)
[D4_prefix] ep 1 step 75/103 loss 2.1424 scale 19.5 (358s)
[D4_prefix] ep 1 step 100/103 loss 2.0794 scale 19.4 (478s)
[D4_prefix] ep 1 train loss 2.0493 | DEV F1@7 0.2924 (P 0.3184 R 0.2703) | R@50 0.673 R@100 0.751 MRR 0.658 (535s)
[D4_prefix] ep 2 step 25/103 loss 1.8814 scale 19.3 (119s)
[D4_prefix] ep 2 step 50/103 loss 1.9423 scale 19.2 (238s)
[D4_prefix] ep 2 step 75/103 loss 1.9024 scale 19.1 (357s)
[D4_prefix] ep 2 step 100/103 loss 1.8427 scale 19.1 (476s)
[D4_prefix] ep 2 train loss 1.8306 | DEV F1@7 0.2979 (P 0.3245 R 0.2754) | R@50 0.701 R@100 0.756 MRR 0.682 (533s)
[D4_prefix] ep 3 step 25/103 loss 1.7816 scale 19.1 (119s)
[D4_prefix] ep 3 step 50/103 loss 1.7910 scale 19.0 (238s)
[D4_prefix] ep 3 step 75/103 loss 1.6700 scale 19.0 (358s)
[D4_prefix] ep 3 step 100/103 loss 1.7104 scale 19.0 (476s)
[D4_prefix] ep 3 train loss 1.7157 | DEV F1@7 0.3002 (P 0.3

{'f1': 0.3001667593107282,
 'p': 0.3268765133171913,
 'r': 0.2774922918807811,
 'r50': 0.7060809624798264,
 'r100': 0.7605162898799863,
 'mrr': 0.6871090142388093}

In [ ]:
run_experiment("D5_partial", method="partial", unfreeze_top_k=3)


=== D5_partial | top 3 encoder layers + final norm | trainable 12.85M / 54.8M ===
[D5_partial] 817 train queries | 103 steps/epoch x 3 | batch 8 queries x (1 pos + 3 hard neg, hn=bm25) | K_train 4 chunks | head 0.26M backbone 12.59M


/tmp/ipykernel_1330/2604840094.py:168: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)


[D5_partial] ep 1 step 25/103 loss 2.6532 scale 19.9 (70s)
[D5_partial] ep 1 step 50/103 loss 2.2401 scale 19.8 (139s)
[D5_partial] ep 1 step 75/103 loss 2.1463 scale 19.5 (208s)
[D5_partial] ep 1 step 100/103 loss 2.0805 scale 19.4 (278s)
[D5_partial] ep 1 train loss 2.0519 | DEV F1@7 0.2957 (P 0.3220 R 0.2734) | R@50 0.676 R@100 0.752 MRR 0.651 (322s)
[D5_partial] ep 2 step 25/103 loss 1.8790 scale 19.3 (69s)
[D5_partial] ep 2 step 50/103 loss 1.9338 scale 19.2 (138s)
[D5_partial] ep 2 step 75/103 loss 1.8816 scale 19.2 (207s)
[D5_partial] ep 2 step 100/103 loss 1.8221 scale 19.2 (277s)
[D5_partial] ep 2 train loss 1.8101 | DEV F1@7 0.2991 (P 0.3257 R 0.2765) | R@50 0.682 R@100 0.752 MRR 0.676 (321s)
[D5_partial] ep 3 step 25/103 loss 1.7327 scale 19.1 (69s)
[D5_partial] ep 3 step 50/103 loss 1.7504 scale 19.1 (138s)
[D5_partial] ep 3 step 75/103 loss 1.6338 scale 19.1 (208s)
[D5_partial] ep 3 step 100/103 loss 1.6732 scale 19.1 (277s)
[D5_partial] ep 3 train loss 1.6768 | DEV F1@7 0

{'f1': 0.29905503057254035,
 'p': 0.32566585956416466,
 'r': 0.27646454265159304,
 'r50': 0.6818781108984696,
 'r100': 0.7520797584092851,
 'mrr': 0.6760992911260085}

In [ ]:
run_experiment("D6_kd",      method="kd", teacher_ckpt=os.path.join(p7_dir, "ckpt_D1_full.pt"))   # student = half the encoder layers

caching teacher representations of the train split ...
student encoder: 5 layers from teacher layers [0, 2, 4, 6, 8] | 37.8M params (teacher 54.5M)

=== D6_kd | student backbone (all weights trainable) | trainable 38.02M / 38.0M ===
[D6_kd] 817 train queries | 103 steps/epoch x 3 | batch 8 queries x (1 pos + 3 hard neg, hn=teacher) | K_train 4 chunks | head 0.26M backbone 37.75M
[D6_kd] ep 1 step 25/103 loss 1.3933 scale 19.2 (68s)
[D6_kd] ep 1 step 50/103 loss 1.3337 scale 19.1 (134s)
[D6_kd] ep 1 step 75/103 loss 1.3008 scale 19.0 (202s)
[D6_kd] ep 1 step 100/103 loss 1.3248 scale 19.0 (269s)
[D6_kd] ep 1 train loss 1.3393 | DEV F1@7 0.2879 (P 0.3136 R 0.2662) | R@50 0.685 R@100 0.760 MRR 0.660 (298s)
[D6_kd] ep 2 step 25/103 loss 1.2047 scale 18.9 (67s)
[D6_kd] ep 2 step 50/103 loss 1.1904 scale 18.9 (133s)
[D6_kd] ep 2 step 75/103 loss 1.2145 scale 18.8 (200s)
[D6_kd] ep 2 step 100/103 loss 1.1966 scale 18.8 (267s)
[D6_kd] ep 2 train loss 1.1861 | DEV F1@7 0.2902 (P 0.3160 R 0.2682

{'f1': 0.292384658143413,
 'p': 0.31840193704600483,
 'r': 0.27029804727646456,
 'r50': 0.698251227669523,
 'r100': 0.7769431677870633,
 'mrr': 0.6689009961588999}